# IPL Feature Engineering Pipeline

This notebook orchestrates the feature engineering pipeline. It compiles ball-by-ball deliveries, standardizes values, and calculates career/rolling stats for batsmen and bowlers, venue conditions, and team form features.

## 1. Setup & Imports

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from config.paths import RAW_DATA_DIR, STAGING_DIR, CRICSHEET_DIR
from src.etl.deliveries import process_deliveries_dataset
from src.etl.feature_builder import (
    build_features,
    compute_historical_batsman_stats,
    compute_historical_bowler_stats
)

## 2. Load Processed Match Data & Catalog

In [3]:
catalog_df = pd.read_csv(STAGING_DIR / "dataset_catalog.csv")
match_df = pd.read_parquet(STAGING_DIR / "match_info_processed.parquet")
print(f"Loaded catalog: {catalog_df.shape}")
print(f"Loaded matches: {match_df.shape}")

Loaded catalog: (1243, 6)
Loaded matches: (1243, 36)


## 3. Process & Compile Deliveries

In [4]:
deliveries_df = process_deliveries_dataset(catalog_df, CRICSHEET_DIR)
print(f"Compiled deliveries shape: {deliveries_df.shape}")

# Save master deliveries to staging
deliveries_output_path = STAGING_DIR / "deliveries_processed.parquet"
deliveries_df.to_parquet(deliveries_output_path, index=False)
print(f"Saved processed deliveries to: {deliveries_output_path}")

Processing deliveries:   0%|          | 0/1243 [00:00<?, ?it/s]

Processing deliveries: 100%|██████████| 1243/1243 [00:04<00:00, 285.32it/s]


Compiled deliveries shape: (295732, 27)
Saved processed deliveries to: D:\IPL-Analytics-Strategy-Platform\data\processed\staging\deliveries_processed.parquet


## 4. Batsman Career & Historical Stats

In [5]:
batsman_features_df = compute_historical_batsman_stats(deliveries_df)
print(f"Batsman features shape: {batsman_features_df.shape}")
display(batsman_features_df.head())

Batsman features shape: (18768, 6)


,match_id,striker,batsman_career_runs,batsman_career_balls,batsman_career_sr,batsman_career_avg
0,335982,AA Noffke,0,0,0.0,0.0
15,335982,Z Khan,0,0,0.0,0.0
13,335982,V Kohli,0,0,0.0,0.0
12,335982,SC Ganguly,0,0,0.0,0.0
11,335982,SB Joshi,0,0,0.0,0.0


## 5. Bowler Career & Historical Stats

In [6]:
bowler_features_df = compute_historical_bowler_stats(deliveries_df)
print(f"Bowler features shape: {bowler_features_df.shape}")
display(bowler_features_df.head())

Bowler features shape: (14700, 6)


,match_id,bowler,bowler_career_wickets,bowler_career_runs,bowler_career_balls,bowler_career_economy
0,335982,AA Noffke,0,0,0,0.0
9,335982,SC Ganguly,0,0,0,0.0
8,335982,SB Joshi,0,0,0,0.0
7,335982,P Kumar,0,0,0,0.0
6,335982,LR Shukla,0,0,0,0.0


## 6. Venue Analytics Features

In [7]:
# Compute actual first innings scores from deliveries
first_innings_runs = (
    deliveries_df[deliveries_df["innings"] == 1]
    .groupby("match_id")
    .agg(runs_off_bat=("runs_off_bat", "sum"), extras=("extras", "sum"))
    .reset_index()
)
first_innings_runs["first_innings_score"] = first_innings_runs["runs_off_bat"] + first_innings_runs["extras"]

# Merge with matches to get venue
venue_scores = first_innings_runs.merge(
    match_df[["match_id", "venue", "winner", "team1", "team2", "toss_winner", "toss_decision"]],
    on="match_id"
)

# Venue average first innings score
venue_avg_scores = (
    venue_scores.groupby("venue")["first_innings_score"]
    .mean()
    .reset_index(name="venue_avg_first_innings_score")
    .round(2)
)

# Chasing Win Identification
def is_chase_win(row):
    if pd.isna(row["winner"]):
        return False
    bat_first = row["toss_winner"] if row["toss_decision"] == "bat" else (
        row["team1"] if row["toss_winner"] == row["team2"] else row["team2"
    ])
    bat_second = row["team2"] if bat_first == row["team1"] else row["team1"]
    return row["winner"] == bat_second

venue_scores["is_chase_win"] = venue_scores.apply(is_chase_win, axis=1)

# Venue chasing win percentage
venue_chase_stats = (
    venue_scores.groupby("venue")["is_chase_win"]
    .mean()
    .reset_index(name="venue_chase_win_pct")
    .round(4)
)
venue_chase_stats["venue_chase_win_pct"] *= 100

# Combine venue features
venue_features_df = venue_avg_scores.merge(venue_chase_stats, on="venue")
print(f"Venue features computed: {venue_features_df.shape}")
display(venue_features_df.head())

Venue features computed: (47, 3)


,venue,venue_avg_first_innings_score,venue_chase_win_pct
0,"Arun Jaitley Stadium, Delhi",185.57,50.00
1,Barabati Stadium,167.71,57.14
2,"Barsapara Cricket Stadium, Guwahati",168.88,50.00
3,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,174.79,55.17
4,"Brabourne Stadium, Mumbai",178.52,48.15


## 7. Merge and Export Final Datasets

In [8]:
# 1. Match-level features
match_features_df = build_features(match_df)
match_features_df = match_features_df.merge(venue_features_df, on="venue", how="left")

match_output_path = STAGING_DIR / "match_features.parquet"
match_features_df.to_parquet(match_output_path, index=False)
print(f"Saved match features to: {match_output_path} (Shape: {match_features_df.shape})")

# 2. Player-level features
# Combine batsman and bowler historical features
player_features_df = batsman_features_df.merge(
    bowler_features_df, 
    left_on=["match_id", "striker"], 
    right_on=["match_id", "bowler"], 
    how="outer"
)
# Combine striker and bowler columns into a single player name column
player_features_df["player_name"] = player_features_df["striker"].fillna(player_features_df["bowler"])
player_features_df = player_features_df.drop(columns=["striker", "bowler"])

player_output_path = STAGING_DIR / "player_features.parquet"
player_features_df.to_parquet(player_output_path, index=False)
print(f"Saved player features to: {player_output_path} (Shape: {player_features_df.shape})")

print("\n========================================")
print("Feature Engineering Pipeline Completed!")
print("========================================")

Saved match features to: D:\IPL-Analytics-Strategy-Platform\data\processed\staging\match_features.parquet (Shape: (1243, 38))
Saved player features to: D:\IPL-Analytics-Strategy-Platform\data\processed\staging\player_features.parquet (Shape: (26703, 10))

Feature Engineering Pipeline Completed!
